In [1]:
pip install -q llama-cpp-python langchain sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")  # Apple Silicon GPU
    print("Using MPS (Apple GPU)")
else:
    device = torch.device("cpu")
    print("Using CPU")


Using MPS (Apple GPU)


In [2]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import LlamaCpp

# 1. Load your FAISS index
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.load_local(
    folder_path="/Users/felixkevinsinght/TN/db_faiss",  # Path where index files are stored
    embeddings=embeddings,
    allow_dangerous_deserialization=True  # Required for FAISS
)

# 2. Load Mistral-7B (4-bit quantized)
llm = LlamaCpp(
    model_path="/Users/felixkevinsinght/TN/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    n_ctx=2048,
    max_tokens=512,
    n_gpu_layers=40,  # Use GPU in Colab
    temperature=0.7,
    verbose=False,
)

/var/folders/r4/b51zz8g55d332tn7jrp2h5hc0000gn/T/ipykernel_4516/1296486906.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
llama_init_from_model: n_batch is less than GGML_KQ_MASK_PAD - increasing to 64
llama_init_from_model: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not support

In [ ]:
from langchain.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Custom prompt template for Mistral
template = """
[INST] <<SYS>>
Use the pieces of information provided in the context to answer user's question.
If you dont know the answer, just say that you dont know, dont try to make up an answer. 
Dont provide anything out of the given context.
If the word limit are given give according to the needs.
Context: {context}
<</SYS>>
Question: {question} [/INST]
"""
prompt = PromptTemplate.from_template(template)

# Create retrieval chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 3}),  # Top 3 relevant chunks
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

# Chat function
def chat():
    print("Bot: Hello! Ask me anything (type 'quit' to exit).")
    while True:
        query = input("\nYou: ")
        if query.lower() == "quit":
            break
        result = qa_chain.invoke({"query": query})
        print(f"\nBot: {result['result']}")
        print("\nSources:", [doc.metadata.get("source", "unknown") for doc in result["source_documents"]])

chat()

Bot: Hello! Ask me anything (type 'quit' to exit).



You:  how to prepare rice cultivation



Bot: To prepare rice cultivation, you need to follow the steps provided in the context. Here is a breakdown of the critical steps:

1. Land Preparation: The first step is to plow the land and then flood it with water for puddling (2.5 cm deep). This helps to loosen the soil and remove any weeds, making it easier for rice plants to grow.
2. Transplanting: Use 4th-leaf seedlings for transplantation. Avoid using aged seedlings as they may not be healthy and productive.
3. Problem Soils: If your land has problem soils such as saline (ZnSO₄ dip) or clayey (gypsum), you need to apply the appropriate solution before planting rice. For saline soil, use a solution of 80 µM (micromolar) Sodium Nitroprusside, and for clayey soil, use gypsum.
4. Nutrient Management: For nutrient management, you can use pre-emergence herbicides such as Butachlor 8 days after sowing (DAS). Additionally, you can apply biofertilizers to improve the health of your rice crops.
5. Seed Treatment and Nursery Management: 